# Deep Learning - Winter Term 2026/2027

<hr style="border:2px solid gray">

### Exercise Sheet 02


# 3) Train MLP on XOR Problem

In the previous exercise sheet we built up our own auto differentiation tool. We implemented many small functions and their derivatives. These functions are more than enough to build a multilayer perceptron (MLP). This is exactly what we are going to do in this exercise. Isn't it cool? Creating your own library and using it to train your first neural network. As in the previous exercise: look for TODOs and replace them with meaningful code.

- We didn't write any tests for you because we want you to get used to make sanity checks, write your own tests and debug your code. These are important skills and you will need them in your future research.
- We suggest you to spend some time with the code base before coding. The exercises do not need to be solved successively. Also, the time you spend to understand will pay off later while coding.
- ***Note:*** You might need to install some of the packages to run the code.

In [66]:
import random

In [67]:
'''
We import our tool from the first exercise.
The reason we keep it in another file is that we
do not need to change it anymore and don't want clutter
in our jupyter notebook.
'''
from CompGraph import Node

### Build up the neural network architecture
In the following, you will create the architecture for the neural network.

In [68]:
class Network:
    'Class for the whole network'
    
    def zero_grad(self):
        '''
        In the training, we compute the gradients many times.
        Thus after each backward pass, we need to clean up
        the gradients to compute the new ones.
        '''
        # TODO: implement an algorithm that zeros the gradients of network parameters.
        # **This goes through all the trainable parameters 
        # (weights and biases) of the neural network
        # This resets the gradient of each parameter to zero.
        for param in self.parameters():
            param.grad = 0.0

    def parameters(self):
        ' Return the parameters '
        return []

In [69]:
class ForwardNeuron(Network):
    'Class computing forward pass via Node'

    def __init__(self, nin: int, linear: bool=False):
        '''
        :param nin: number of input neurons
        :param linear: switch to decide the linearity of the activation function
        
        Here, the weights are initialized. In order to do that, we need to
        know how many weights and biases a feedforward neural network has.
        Each weight and bias is a Node object. We initialize random weights
        uniformly sampled from [-0.5,0.5].
        Please also wrap them up in a list because we work only with lists in our implementation.
        In Pytorch, we will work with tensors.
        '''
        # TODO: Initialize weights and biases
        self.w = [Node(random.uniform(-0.5, 0.5)) for _ in range(nin)]
        self.b = [Node(random.uniform(-0.5, 0.5))]
        self.linear = linear

    def __call__(self, x: list):
        '''
        This method is used when ForwardNeuron class is *called*
        Here, the operation to forward propagate
        information from one layer to the next one is implemented.
        The general formula is $w^Tx$ but here the scalar valued function
        is implemented because CompGraph works only with scalars.
        '''
        # TODO: implement $net = w^Tx$ _and_ apply your choice of non-linearity
        #       or the identity activation function depending on self.linear
        # *Note*: we are working with lists of scalars here
        net = sum(w_i * x_i for w_i, x_i in zip(self.w, x)) + self.b[0]
        if self.linear:
            return net
        else:
            # Apply a non-linear activation function, e.g., ReLU
            return net.relu()

    def parameters(self):
        # TODO: return the parameters of the ForwardNeuron class (weights, bias)
        #       in form of a list of Nodes.
        return self.w + self.b
        raise NotImplementedError

    def __repr__(self):
        # Informs about the linearity of the act. fnc. and the number of neurons from
        # the **prev** layer. It follows from the fact that in a fully connected network,
        # a neuron receives information from each neuron from the previous layer.
        return f"{'Non-linear' if not self.linear else 'Linear'}ForwardNeuron({len(self.w)})"

In [70]:
neuron = ForwardNeuron(2)

neuron.w = [Node(0.4), Node(-0.3)]
neuron.b = [Node(0.1)]

print("ReLU","(w_i*x_i+b)=",neuron([1.0, 0.0]).scalar)
print("ReLU","(w_i*x_i+b)=",neuron([0.0, 1.0]).scalar)
print("Number of parameters:", len(neuron.parameters()))

ReLU (w_i*x_i+b)= 0.5
ReLU (w_i*x_i+b)= 0.0
Number of parameters: 3


In [71]:
class Layer(Network):
    'Class creating layer objects via ForwardNeuron'

    def __init__(self, nin, nout, linear):
        self.neurons = [ForwardNeuron(nin, linear) for _ in range(nout)]

    def __call__(self, x):
        return [n(x) for n in self.neurons]

    def parameters(self):
        # TODO: return the parameters of the entire layer (for all neurons) in form
        #       of list of nodes.
        return [param for neuron in self.neurons for param in neuron.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

In [72]:
class MLP(Network):
    'Class creating MLP object via Layer'

    def __init__(self, nin, nouts):
        # Concatenates two lists to compute the size.
        size = [nin] + nouts

        # Set the layers
        # It might be helpful to understand following line good. It is a condensed coding.
        self.layers = [Layer(size[i], size[i+1], linear=(i == len(nouts)-1)) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            x = layer(x)
        return x[0] if len(x) == 1 else x

    def parameters(self):
        #TODO: return all parameters in MLP in form of a list of Nodes.
        return [param for layer in self.layers for param in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

In [73]:

model = MLP(2, [3, 1])
print("2-3-1 MLP:   P=2*3=3*1=(3+1)=13")
print(len(model.parameters()))

2-3-1 MLP:   P=2*3=3*1=(3+1)=13
13


### XOR problem

In [74]:
# Create the XOR problem (data and target)
X = [
    [0.0, 0.0],
    [0.0, 1.0],
    [1.0, 0.0],
    [1.0, 1.0]]

y = [0.0, 1.0, 1.0, 0.0] # targets

# SGD with momentum

In [75]:
# Set some seeds for reproducibility
for n in range(1,21):
    random.seed(4242)
    model = MLP(2, [n, 1])

    eta = 0.15
    mu = 0.9
    epochs = 1000
    params = model.parameters()
    momentum = [0.0 for p in params]
    
# training loop
#TODO: implement the training algorithm. Here are the steps you should follow:
    for epoch in range(epochs):


#(1) Zero the gradients (this is a must, otherwise there won't be any learning)
        model.zero_grad()
#(2) Forward pass (don't forget to wrap the outputs in a list)
        yhat = [model(x) for x in X]
#(3) Compute the L2 loss
        loss = sum((yhat_i - y_i) ** 2 for yhat_i, y_i in zip(yhat, y)) / len(y)
#(4) Backward pass
        loss.backward()
#(5) Weight update using gradient descent with momentum (bonus task: Nesterov momentum)
        for i, p in enumerate(params):
            # Update momentum
            momentum[i] = mu * momentum[i] - eta * p.grad
            # Update parameter
            p.scalar += momentum[i]

#**Note** You might want to print the loss to follow what the network is doing.
    predictions = [model(x) for x in X]
    print("hidden layer size:", n, "Predictions:", [round(p.scalar, 4) for p in predictions])

    if all(abs(p.scalar - t) < 0.1 for p, t in zip(predictions, y)):
        print("The network has learned the XOR function with hidden layer size:", n)
        break

print()

hidden layer size: 1 Predictions: [0.5, 0.5, 0.5, 0.5]
hidden layer size: 2 Predictions: [0.5, 0.5, 0.5, 0.5]
hidden layer size: 3 Predictions: [0.5, 0.5, 0.5, 0.5]
hidden layer size: 4 Predictions: [0.5, 0.5, 0.5, 0.5]
hidden layer size: 5 Predictions: [0.5, 0.5, 0.5, 0.5]
hidden layer size: 6 Predictions: [0.0, 0.6667, 0.6667, 0.6667]
hidden layer size: 7 Predictions: [0.0, 0.6667, 0.6667, 0.6667]
hidden layer size: 8 Predictions: [0.0, 0.5, 1.0, 0.5]
hidden layer size: 9 Predictions: [0.0, 0.5, 1.0, 0.5]
hidden layer size: 10 Predictions: [0.3333, 0.3333, 1.0, 0.3333]
hidden layer size: 11 Predictions: [0.0, 1.0, 1.0, 0.0]
The network has learned the XOR function with hidden layer size: 11



In [76]:

yhat = [model(x) for x in X]

for x, yt, yp in zip(X, y, yhat):
    print("Input:", x,
          "Target:", yt,
          "Prediction:", round(yp.scalar, 4))
# You can look at the model predictions and check if they are correct:
# print(yhat) or print(y_hat) or whatever how you define the model prediction

Input: [0.0, 0.0] Target: 0.0 Prediction: 0.0
Input: [0.0, 1.0] Target: 1.0 Prediction: 1.0
Input: [1.0, 0.0] Target: 1.0 Prediction: 1.0
Input: [1.0, 1.0] Target: 0.0 Prediction: 0.0


# Bonus task

## Nesterov classical implementation

$$
\begin{align}
\hat{\mathbf{w}}^{t+1} &= \mathbf{w}^{t} - \eta \mu \mathbf{m}^{t}\\
\mathbf{m}^{t+1} &= \mu \mathbf{m}^{t} + (1 - \mu) \nabla_{\mathbf{w}} \mathcal{L}(\hat{\mathbf{w}}^{t+1})\\
\mathbf{w}^{t+1} &= \mathbf{w}^{t} - \eta \mathbf{m}^{t+1}
\end{align}
$$


In [77]:

# Set some seeds for reproducibility
random.seed(4242)

# Initialize the model
model = MLP(2, [11, 1])
print(model)

# Training parameters
eta = 0.15
mu = 0.9
epochs = 1000

params = model.parameters()
momentum = [0.0 for p in params]

# training loop
for epoch in range(epochs):

    # Save the original parameters
    old_params = [p.scalar for p in params]
    # Calculate the look-ahead position
    for i, p in enumerate(params):
        p.scalar = old_params[i] - eta * mu * momentum[i]

    # (1) Zero the gradients
    model.zero_grad()
    # (2) Forward pass
    yhat = [model(x) for x in X]
    # (3) Compute the L2 loss
    loss = sum((yhat_i - y_i)**2
               for yhat_i, y_i in zip(yhat, y)) / len(y)
    # (4) Backward pass
    loss.backward()
    # (5) Weight update using Nesterov momentum
    for i, p in enumerate(params):

        # Update momentum
        momentum[i] = mu * momentum[i] + (1 - mu) * p.grad
        # Update parameter using the original value
        p.scalar = old_params[i] - eta * momentum[i]

    # Print the loss
    if epoch % 100 == 0:
        print("Epoch:", epoch, "Loss:", loss.scalar)
print()


MLP of [Layer of [Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2), Non-linearForwardNeuron(2)], Layer of [LinearForwardNeuron(11)]]
Epoch: 0 Loss: 0.26299350362125545
Epoch: 100 Loss: 0.002557445471143405
Epoch: 200 Loss: 1.1225357119344205e-07
Epoch: 300 Loss: 6.810677498111344e-13
Epoch: 400 Loss: 3.1091630045321537e-19
Epoch: 500 Loss: 3.506311646130796e-24
Epoch: 600 Loss: 1.7306711244162028e-28
Epoch: 700 Loss: 2.0701268799512506e-31
Epoch: 800 Loss: 1.0167390596732409e-31
Epoch: 900 Loss: 9.871381964864282e-32

